# POC — Ingestão de CSV para MySQL
### Tabela: `tb_dados_brutos`
---
**Stack:** Python · Pandas · SQLAlchemy · mysql-connector-python  
**Pré-requisitos:**
```bash
pip install pandas sqlalchemy mysql-connector-python
```

## 1. Imports

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import time

## 2. Configurações de conexão

In [13]:
# ── Configurações ──────────────────────────────────────────────
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_USER     = 'root'                 # altere para seu usuário
DB_PASSWORD = '123456'               # altere para sua senha
DB_NAME     = 'db_poc_call_center'   # altere se necessário

from pathlib import Path

# CSV_PATH = (
#     Path.cwd()          # D:\Git-Projetos\Python\POC-Call-Center\notebooks
#     .parent             # D:\Git-Projetos\Python\POC-Call-Center
#     .parent             # D:\Git-Projetos\Python
#     .parent             # D:\Git-Projetos
#     .parent             # D:\                          ← raiz comum
#     / 'Users' / 'rtoni' / 'OneDrive' / 'Git-Dados' / 'POC-Call-Center'
#     / 'tb_ligacao_202602271123.csv'
#     # / 'Tabela_DDD.csv'
# )

# CSV_PATH: Path = Path(r'C:\Users\rtoni\OneDrive\Git-Dados\POC-Call-Center\tb_ligacao_202602271123.csv')
CSV_PATH: Path = Path(r'C:\Users\rtoni\OneDrive\Git-Dados\POC-Call-Center\Tabela_DDD.csv')

# TABLE_NAME  = 'tb_dados_bruto'
TABLE_NAME  = 'tb_ddd_brasil'
SEPARATOR   = ';'             # separador do CSV
CHUNK_SIZE  = 500             # linhas por lote (ajuste conforme volume)
# ───────────────────────────────────────────────────────────────

## 3. Criando a engine de conexão

In [9]:
connection_string = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(connection_string, echo=False)

# Teste de conexão
with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Conexão OK!' AS status"))
    print(result.fetchone()[0])

Conexão OK!


## 4. Leitura e validação do CSV

In [14]:
df = pd.read_csv(
    CSV_PATH,
    sep=SEPARATOR,
    dtype=str,            # lê tudo como texto (espelha VARCHAR no MySQL)
    keep_default_na=False # mantém campos vazios como string vazia
)

# Normaliza nomes de coluna (remove espaços extras, se houver)
df.columns = df.columns.str.strip()

print(f"✅ Arquivo lido com sucesso!")
print(f"   Linhas   : {len(df):,}")
print(f"   Colunas  : {len(df.columns)}")
print()
df.head(3)

✅ Arquivo lido com sucesso!
   Linhas   : 67
   Colunas  : 3



,Prefixo,Estado,Cidades_Regioes
0,99,Maranhão,Caxias/Codó/Imperatriz
1,98,Maranhão,São Luís e Região Metropolitana
2,97,Amazonas,Abrangência no interior do estado


## 5. Validação das colunas x tabela MySQL

In [15]:
EXPECTED_COLUMNS = [
    'Prefixo', 'Estado','Cidades_Regioes'
    # 'id_ligacao', 'id_dsc_gravacao', 'id_dsc_ticket', 'id_lead',
    # 'nu_dsc_telefone_to', 'nm_gp_rota', 'dh_inicio', 'dh_fim',
    # 'dh_inicio_tabulacao', 'qt_tempo_duracao', 'qt_tempo_falando',
    # 'qt_tempo_tabulando', 'qt_aguardando_chamada', 'id_dsc_contato',
    # 'tp_dsc_contato', 'id_dsc_campanha', 'nm_dsc_campanha', 'id_dsc_lista',
    # 'nm_dsc_lista', 'id_dsc_mailing', 'nm_dsc_mailing', 'id_dsc_status',
    # 'nm_dsc_status', 'nm_dsc_grupo', 'nm_sub_grupo', 'id_dsc_tabulacao',
    # 'nm_dsc_tabulacao', 'id_supervisor', 'nm_supervisor', 'id_dsc_usuario',
    # 'nm_dsc_usuario', 'bl_atendido', 'bl_abordagem', 'bl_target', 'bl_venda',
    # 'bl_agendamento', 'bl_agendamento_pessoal', 'bl_telefone_finalizado',
    # 'bl_finaliza_lead', 'bl_pre_venda', 'bl_recusa', 'bl_improdutivo',
    # 'bl_auditor', 'bl_auditoria_backoffice'
]

missing = set(EXPECTED_COLUMNS) - set(df.columns)
extra   = set(df.columns) - set(EXPECTED_COLUMNS)

if missing:
    print(f"⚠️  Colunas FALTANDO no CSV : {missing}")
if extra:
    print(f"⚠️  Colunas EXTRAS no CSV  : {extra}")
if not missing and not extra:
    print("✅ Colunas validadas — CSV compatível com a tabela MySQL!")

✅ Colunas validadas — CSV compatível com a tabela MySQL!


## 6. Ingestão por lotes (chunked insert)

In [16]:
total_rows   = len(df)
total_chunks = (total_rows // CHUNK_SIZE) + (1 if total_rows % CHUNK_SIZE else 0)
rows_inserted = 0
start_time   = time.time()

print(f"🚀 Iniciando ingestão — {total_rows:,} linhas em {total_chunks} lote(s)...\n")

for i, chunk_start in enumerate(range(0, total_rows, CHUNK_SIZE), start=1):
    chunk = df.iloc[chunk_start : chunk_start + CHUNK_SIZE]

    chunk.to_sql(
        name      = TABLE_NAME,
        con       = engine,
        if_exists = 'append',   # nunca recria a tabela
        index     = False,
        method    = 'multi'     # insert em bloco (mais rápido)
    )

    rows_inserted += len(chunk)
    pct = (rows_inserted / total_rows) * 100
    print(f"   Lote {i:>3}/{total_chunks} — {rows_inserted:>6,}/{total_rows:,} linhas inseridas ({pct:.1f}%)")

elapsed = time.time() - start_time
print(f"\n✅ Ingestão concluída em {elapsed:.2f}s — {rows_inserted:,} linhas gravadas em `{TABLE_NAME}`")

🚀 Iniciando ingestão — 67 linhas em 1 lote(s)...

   Lote   1/1 —     67/67 linhas inseridas (100.0%)

✅ Ingestão concluída em 0.40s — 67 linhas gravadas em `tb_ddd_brasil`


## 7. Validação pós-carga

In [ ]:
with engine.connect() as conn:
    count_db  = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    count_csv = len(df)

print(f"📊 Linhas no CSV           : {count_csv:,}")
print(f"📊 Linhas no MySQL         : {count_db:,}")

if count_db == count_csv:
    print("✅ Contagem OK — todos os registros foram inseridos!")
else:
    diff = count_csv - count_db
    print(f"⚠️  Divergência de {diff:,} linhas — verifique os logs.")

## 8. Preview dos dados no banco

In [ ]:
df_check = pd.read_sql(
    f"SELECT * FROM {TABLE_NAME} LIMIT 5",
    con=engine
)

print(f"Preview — primeiras 5 linhas de `{TABLE_NAME}`:")
df_check